In [1]:
import pandas as pd

no_call = pd.read_csv("ecoh_no_call_by_cluster.tsv", sep="\t")

In [2]:
no_call = no_call[no_call.reason == "no_ecoh_hits"]

In [3]:
serotypes = pd.read_csv("oag_locus_serotype.csv")

In [4]:
set(serotypes.cluster) & set(no_call.cluster)

{22, 39, 139}

Cluster 39: Some genes have an IS5 element which have deleted the wzx/wzy genes for O114.
Cluster 139: 90% identity to Onovel15 isolate across entire locus
Cluster 22: Some similarity to O108 but genes in the middle missing


In [5]:
unique_clusters = list(set(no_call.cluster))
unique_clusters.remove(22)
unique_clusters.remove(39)
unique_clusters.remove(139)

In [6]:
from Bio import SeqIO
genbank_files = {}
i = 0
# Iterate through records in a file
for record in SeqIO.parse("O-antigen_db.reverse_complement.is_removed.no_transposase.gbff", "genbank"):
    likely_capsule = False
    for feature in record.features:
        if feature.type == "CDS":
            if "wzi" in feature.qualifiers["product"][0].lower():
                likely_capsule = True
                i += 1
            elif "wza" in feature.qualifiers["product"][0].lower():
                likely_capsule = True
            elif "serotype k" in feature.qualifiers["product"][0].lower():
                likely_capsule = True
                i +=1
    if not likely_capsule:
        genbank_files[record.id] = record

In [7]:
novel_otypes = []
for cluster in unique_clusters:
    locus = no_call[no_call.cluster == cluster]
    locus_name = locus.iloc[0].locus
    if genbank_files.get(locus_name) is not None:
        record = genbank_files[locus_name]
        for feature in record.features:
            if feature.type == "source":
                feature.qualifiers["serotype"] = [f"Ois{cluster}"]

        novel_otypes.append(record)
        SeqIO.write(record, f"Novel_Otypes_rep/{record.id}.gbk", "genbank")
SeqIO.write(novel_otypes, "Novel_Otypes_rep/combined_novel_otypes.gbff", "genbank")

/Users/uqsmor23/miniconda3/envs/ECRefSeq_Oag_db/lib/python3.13/site-packages/Bio/SeqIO/InsdcIO.py:779: BiopythonWarning: Increasing length of locus line to allow long name. This will result in fields that are not in usual positions.
  warnings.warn(


61

In [8]:
no_call.cluster.value_counts()

cluster
198    13
132    12
19     11
179     9
3       9
       ..
258     1
44      1
150     1
177     1
85      1
Name: count, Length: 70, dtype: int64